# Out-of-Distribution Generalization Suite (LOGO & LODO)
## Leave-One-Generator-Out & Leave-One-Domain-Out Retraining Benchmark

This notebook trains `roberta-base` from scratch across 3 random seeds (42, 43, 44) holding out:
- **Generators:** Claude, Gemini, GPT-5, GPT-OSS-120B, ChatGPT, Llama
- **Domains:** Academic, Essay, QA

Decision thresholds are tuned strictly on `val.parquet` of the remaining training groups (Rule 4), and evaluated on held-out slices of `test.parquet`.

In [ ]:
# 1. Mount Google Drive and navigate to project repository
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    for candidate in [
        '/content/drive/MyDrive/AI_Text_Checker',
        '/content/drive/MyDrive/AI-text-checker',
        '/content/drive/Shareddrives/AI_Text_Checker'
    ]:
        if os.path.exists(candidate):
            os.chdir(candidate)
            print(f'Active working directory: {candidate}')
            break
except ImportError:
    pass

print(f'Current directory: {os.getcwd()}')
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow


In [ ]:
# 2. Run unconstrained full-dataset Leave-One-Out Retraining across all seeds
# Note: On a T4/A100 GPU, full training runs efficiently with fp16 autocast.
!python eval/run_ood_leave_one_out_colab.py --seeds 42 43 44 --epochs 2 --batch_size 32


In [ ]:
# 3. Display Final Generalization Results
import pandas as pd
if os.path.exists('results/ood_leave_one_out.csv'):
    df = pd.read_csv('results/ood_leave_one_out.csv')
    display(df[['held_out_target', 'group_type', 'n_ai', 'n_human', 'heldout_recall_mean', 'heldout_fpr_mean', 'delta_recall', 'delta_fpr', 'generalization_verdict']])
